# Setup

In [1]:
import json
import re
import sys
import unicodedata
from pathlib import Path

### The sample corpus

Everyone works on the same two documents, so that every number you see in this
notebook is reproducible:

| File | What it is | Why it is here |
|---|---|---|
| `ai_course.pdf` | 4-page course notes on ML, transformers, attention, embeddings, RAG | a **PDF**: pages, footers, hard line breaks |
| `documentation.txt` | FAISS handbook + our internal notes | a **text file**: ASCII banners, code blocks, ragged blank lines |


# Document Loading 

{
    "text':"MAchine learning is a field of AI....",
    "metadata":{
        "source":"ai_course_pdf",
        "page":1
    }
}

In [2]:
def load_txt(path):
    path= Path(path)
    
    text = path.read_text(encoding="utf-8", errors="replace")
    
    return [{
        "text":text, 
        "metadata":{"source":path.name,
                    "path":str(path), 
                    "page":None, 
                    "type":"text"},
    }]

In [3]:
txt_docs = load_txt("data/documents/documentation.txt")
print ("documents from the .txt file: ",len(txt_docs))
print ("metadata:", txt_docs[0]["metadata"])

documents from the .txt file:  1
metadata: {'source': 'documentation.txt', 'path': 'data\\documents\\documentation.txt', 'page': None, 'type': 'text'}


In [4]:
print("characters:", len(txt_docs[0]["text"]))
print("------First300 characters------")
print(txt_docs[0]["text"][:300])

characters: 11679
------First300 characters------
FAISS - Vector Index Handbook
Internal engineering notes  ::  v2.4  ::  Last updated: March 2026


1. OVERVIEW

FAISS   (Facebook AI Similarity Search) is a C++ library with Python bindings
fo


## PDFs: One document per page

In [5]:
from pypdf import PdfReader 

In [6]:
def load_pdf(path):
    path = Path(path)
    
    reader= PdfReader(str(path))
    documents = []
    
    for page_nb, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if not text.strip():
            continue 
        
        documents.append({
            "text":text, 
            "metadata":{"source":path.name,
                        "path":str(path),
                        "page":page_nb, 
                        "type":"pdf"}
        })
        
    return documents    

In [7]:
pdf_docs = load_pdf("data/documents/ai_course.pdf")

print(f"{len(pdf_docs)} page loaded ")

for doc in pdf_docs: 
    first_line= doc["text"].strip().split("\n")[0]
    print (f"page {doc['metadata']['page']}: {len(doc['text'])} chars | {first_line[:55]}")

4 page loaded 
page 1: 2458 chars | AI Builders Bootcamp - Course Notes
page 2: 2333 chars | Chapter 3: Neural Networks
page 3: 2472 chars | Chapter 5: The Attention Mechanism
page 4: 2146 chars | Chapter 7: Retrieval Augmented Generation


### One loader to dispatch them all


In [8]:
SUPPORTED_TEXT = {".txt",".md"}
SUPPORTED_PDF= {".pdf"}

def load_documents(folder, recursive = True):
    folder= Path(folder)
    documents= []
    
    for path in sorted(folder.glob("**/*" if recursive else "*")): 
        if not path.is_file():
            continue
        
        suffix = path.suffix.lower()
        if suffix in SUPPORTED_TEXT: 
            documents.extend(load_txt(path))
        elif suffix in SUPPORTED_PDF: 
            documents.extend(load_pdf(path)) 
        else: 
            print (f" (skipped unsupported file: {path.name})") 
            
    return documents               

In [9]:
raw_documents = load_documents("data/documents")
print(f" {len(raw_documents)} documents")
print (f"{'source':<22} {'page':>6} {'chara':>9} ")
for doc in raw_documents: 
    meta = doc['metadata']
    print(f"{meta['source']:<22} {str(meta['page'] or '-'):>6} {len(doc['text']):>9} ")

 5 documents
source                   page     chara 
ai_course.pdf               1      2458 
ai_course.pdf               2      2333 
ai_course.pdf               3      2472 
ai_course.pdf               4      2146 
documentation.txt           -     11679 


## 2. Text cleaning


In [10]:
x = "Hello\nWorld"
print(x)
print (str(x))
print(repr(x))

Hello
World
Hello
World
'Hello\nWorld'


In [11]:
page_3 = pdf_docs[2]["text"]
print("RAW PDF TEXT (repr, so newlines are visible)\n")
print(repr(page_3[:420]))
print("\n\nRAW TXT TEXT\n")
print(repr(txt_docs[0]["text"][:300]))

RAW PDF TEXT (repr, so newlines are visible)

'Chapter 5: The Attention Mechanism\nAttention is the operation that lets a model decide, for every token, which other\ntokens matter. Each token is projected into three vectors: a query, a key and a\nvalue. The relevance of token j to token i is the dot product between the query of i\nand the key of j. Those scores are divided by the square root of the head dimension\nto keep them numerically stable, passed through a soft'


RAW TXT TEXT

'FAISS - Vector Index Handbook\nInternal engineering notes  ::  v2.4  ::  Last updated: March 2026\n\n\n=====================================================\n1. OVERVIEW\n=====================================================\n\nFAISS   (Facebook AI Similarity Search) is a C++ library with Python bindings\nfo'


In [12]:
BOILERPLATE_PATTERNS = [
    re.compile(r"^\s*page\s+\d+(\s+of\s+\d+)?\s*$", re.IGNORECASE),  # "Page 3 of 4"
    re.compile(r"^\s*[-=_*~]{3,}\s*$"),                              # "========"
    re.compile(r"^\s*\d+\s*$"),                                      # a lone page number
]

def is_boilerplate(line):
    return any(pattern.match(line) for pattern in BOILERPLATE_PATTERNS)


def clean_text(text, join_wrapped_lines=True, drop_boilerplate=True):
    """Raw extracted text -> text worth embedding."""
    # 1. unicode normalisation
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00a0", " ")                     # non-breaking space
    text = text.replace("\r\n", "\n").replace("\r", "\n")     # Windows / old Mac endings

    # 2. words hyphenated across a line break
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # 3. drop boilerplate, squeeze intra-line whitespace
    lines = []
    for line in text.split("\n"):
        line = re.sub(r"[ \t]+", " ", line).strip()
        if drop_boilerplate and is_boilerplate(line):
            continue
        lines.append(line)
    text = "\n".join(lines)

    # 4. rebuild paragraphs (placeholder trick: protect the real breaks first)
    if join_wrapped_lines:
        text = re.sub(r"\n{2,}", "<PARA>", text)
        text = text.replace("\n", " ")
        text = text.replace("<PARA>", "\n\n")

    # 5. final squeeze
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


cleaned_page_3 = clean_text(page_3)

print("BEFORE\n" + "-" * 70)
print(repr(page_3[:300]))
print("\nAFTER\n" + "-" * 70)
print(repr(cleaned_page_3[:300]))
print("\nREADABLE\n" + "-" * 70)
print(cleaned_page_3[:400])
print(f"\n{len(page_3)} chars -> {len(cleaned_page_3)} chars "
      f"({100 * (1 - len(cleaned_page_3) / len(page_3)):.1f}% removed)")

BEFORE
----------------------------------------------------------------------
'Chapter 5: The Attention Mechanism\nAttention is the operation that lets a model decide, for every token, which other\ntokens matter. Each token is projected into three vectors: a query, a key and a\nvalue. The relevance of token j to token i is the dot product between the query of i\nand the key of j. '

AFTER
----------------------------------------------------------------------
'Chapter 5: The Attention Mechanism Attention is the operation that lets a model decide, for every token, which other tokens matter. Each token is projected into three vectors: a query, a key and a value. The relevance of token j to token i is the dot product between the query of i and the key of j. '

READABLE
----------------------------------------------------------------------
Chapter 5: The Attention Mechanism Attention is the operation that lets a model decide, for every token, which other tokens matter. Each token is project

### Experiment 2.1 — cleaning is not one-size-fits-all


In [13]:
snippet_start = txt_docs[0]["text"].find("index.add(vectors)")
snippet = txt_docs[0]["text"][snippet_start - 120:snippet_start + 260]

print("ORIGINAL\n" + "-" * 70)
print(snippet)
print("\nclean_text(join_wrapped_lines=True)   <- prose setting\n" + "-" * 70)
print(clean_text(snippet, join_wrapped_lines=True))
print("\nclean_text(join_wrapped_lines=False)  <- structure-preserving setting\n" + "-" * 70)
print(clean_text(snippet, join_wrapped_lines=False))

ORIGINAL
----------------------------------------------------------------------
ses a type error or silently copies.

    import numpy as np

    vectors = np.asarray(embeddings, dtype="float32")
    index.add(vectors)
    print(index.ntotal)          # how many vectors are in the index

FAISS assigns sequential integer ids: the first vector added is 0, the second
is 1, and so on.  Row i of the index is row i of your chunk list.  Keep those
two aligned or 

clean_text(join_wrapped_lines=True)   <- prose setting
----------------------------------------------------------------------
ses a type error or silently copies.

import numpy as np

vectors = np.asarray(embeddings, dtype="float32") index.add(vectors) print(index.ntotal) # how many vectors are in the index

FAISS assigns sequential integer ids: the first vector added is 0, the second is 1, and so on. Row i of the index is row i of your chunk list. Keep those two aligned or

clean_text(join_wrapped_lines=False)  <- structure-preserv

In [14]:
def clean_documents(documents):
    """Clean each Document with the settings appropriate to its type."""
    cleaned = []
    for doc in documents:
        is_pdf = doc["metadata"].get("type") == "pdf"
        text = clean_text(doc["text"], join_wrapped_lines=is_pdf)
        if text:                                   # drop documents that cleaned to nothing
            cleaned.append({"text": text, "metadata": dict(doc["metadata"])})
    return cleaned


documents = clean_documents(raw_documents)

before = sum(len(d["text"]) for d in raw_documents)
after = sum(len(d["text"]) for d in documents)
print(f"{len(documents)} documents, {before} -> {after} characters "
      f"({100 * (1 - after / before):.1f}% removed)")

5 documents, 21088 -> 18724 characters (11.2% removed)


## 3. Chunking


In [18]:
# A rough sense of scale. 1 token ~ 4 characters of English.
total_chars = sum(len(d["text"]) for d in documents)
approx_tokens = total_chars / 4

print(f"corpus: {total_chars:,} characters  ~= {approx_tokens:,.0f} tokens")
print(f"a 500-char chunk    ~= {500 / 4:.0f} tokens")
print(f"4 chunks of context ~= {4 * 500 / 4:.0f} tokens  <- what we will actually send\n")

for name, window in [("GPT-3.5 (2023)", 4_096), ("typical 2026 model", 128_000)]:
    print(f"{name:<20} {window:>8,} tokens -> our corpus uses "
          f"{100 * approx_tokens / window:5.1f}% of it")
print("\nNow imagine 10,000 documents instead of 2.")

corpus: 18,724 characters  ~= 4,681 tokens
a 500-char chunk    ~= 125 tokens
4 chunks of context ~= 500 tokens  <- what we will actually send

GPT-3.5 (2023)          4,096 tokens -> our corpus uses 114.3% of it
typical 2026 model    128,000 tokens -> our corpus uses   3.7% of it

Now imagine 10,000 documents instead of 2.


### Fixed-size chunking


In [19]:
def chunk_text_no_overlap(text, chunk_size=500):
    """Naive version: cut every chunk_size characters. Watch the boundaries."""
    return [text[i:i + chunk_size].strip() for i in range(0, len(text), chunk_size)]


demo_text = documents[0]["text"]          # page 1 of the PDF
naive_chunks = chunk_text_no_overlap(demo_text, chunk_size=500)

print(f"{len(demo_text)} chars -> {len(naive_chunks)} chunks\n")
print("END of chunk 0:")
print("   ..." + naive_chunks[0][-90:])
print("\nSTART of chunk 1:")
print("   " + naive_chunks[1][:90] + "...")
print("\n^ The sentence that spans the boundary now exists in neither chunk as a")
print("  complete thought. Whichever chunk retrieval picks, the answer is cut in half.")

2446 chars -> 5 chunks

END of chunk 0:
   ...our from data instead of from explicit rules written by a developer. A traditional program

START of chunk 1:
   encodes the rules directly; a machine learning system is shown examples and derives the ru...

^ The sentence that spans the boundary now exists in neither chunk as a
  complete thought. Whichever chunk retrieval picks, the answer is cut in half.


### Overlapping chunks


In [20]:
def chunk_text(text, chunk_size=500, chunk_overlap=50):
    """Fixed-size character chunks with overlap.

    The window advances by (chunk_size - chunk_overlap), so consecutive chunks
    share their boundary text and no sentence falls between two chunks.
    """
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if chunk_overlap < 0:
        raise ValueError("chunk_overlap cannot be negative")
    if chunk_overlap >= chunk_size:
        # step would be <= 0 -> the loop would never advance -> infinite loop.
        # Try it with chunk_overlap=chunk_size after removing this guard. Once.
        raise ValueError("chunk_overlap must be smaller than chunk_size")

    chunks = []
    start, length = 0, len(text)
    while start < length:
        end = start + chunk_size
        piece = text[start:end].strip()
        if piece:                       # a whitespace-only window adds nothing
            chunks.append(piece)
        if end >= length:               # we just took the tail; stop
            break
        start = end - chunk_overlap     # <- step back by the overlap
    return chunks


overlapped = chunk_text(demo_text, chunk_size=500, chunk_overlap=50)

print(f"no overlap : {len(naive_chunks)} chunks")
print(f"overlap 50 : {len(overlapped)} chunks   (more chunks, same text)\n")
print("END of chunk 0:")
print("   ..." + overlapped[0][-90:])
print("\nSTART of chunk 1:")
print("   " + overlapped[1][:90] + "...")

shared = overlapped[0][-50:].strip()
print(f"\nshared text present in both chunks: {shared[:60]!r}")
print("in chunk 1?", shared[:30] in overlapped[1])

no overlap : 5 chunks
overlap 50 : 6 chunks   (more chunks, same text)

END of chunk 0:
   ...our from data instead of from explicit rules written by a developer. A traditional program

START of chunk 1:
   ules written by a developer. A traditional program encodes the rules directly; a machine l...

shared text present in both chunks: 'ules written by a developer. A traditional program'
in chunk 1? True


### Carrying metadata down to the chunk


Chunk1: 
TExt:"Machine LEarning is a subfield of AI"
Metadata: 
source: ai_course.pdf
page=5

Chunk2: 
TExt:"It has multiple types: supervised, unsupervised, semi-supervised...."
Metadata: 
source: ai_course.pdf
page=5
chunl = 2 
ai_course.pdf#c2


In [21]:
def chunk_documents(documents, chunk_size=500, chunk_overlap=50, chunker=chunk_text):
    """Chunk a list of Documents, propagating metadata to every chunk."""
    all_chunks = []
    for doc in documents:
        pieces = chunker(doc["text"], chunk_size, chunk_overlap)
        for index, piece in enumerate(pieces):
            metadata = dict(doc["metadata"])       # copy! never mutate the parent
            metadata["chunk_index"] = index
            metadata["n_chunks"] = len(pieces)
            metadata["n_chars"] = len(piece)
            source, page = metadata.get("source", "unknown"), metadata.get("page")
            chunk_id = f"{source}#p{page}#c{index}" if page else f"{source}#c{index}"
            all_chunks.append({"id": chunk_id, "text": piece, "metadata": metadata})
    return all_chunks


chunks = chunk_documents(documents, chunk_size=500, chunk_overlap=50)

print(f"{len(documents)} documents -> {len(chunks)} chunks\n")
print(json.dumps(chunks[7], indent=2)[:700])

5 documents -> 44 chunks

{
  "id": "ai_course.pdf#p2#c1",
  "text": "forward pass computes the prediction and the loss. The backward pass then applies the chain rule of calculus layer by layer, from the output back to the input, to obtain the gradient of the loss with respect to every weight. The optimiser, usually a variant of stochastic gradient descent such as Adam, updates the weights using those gradients. - Feedforward networks pass information in one direction only. - Convolutional networks share weights across space and excel at images. - Recurrent networ",
  "metadata": {
    "source": "ai_course.pdf",
    "path": "data\\documents\\ai_course.pdf",
    "page": 2,
    "type": "pdf",
    "chunk_index": 1,
    


In [22]:
# Sanity checks worth running on any corpus before you spend money embedding it.
lengths = [len(c["text"]) for c in chunks]
sources = {}
for chunk in chunks:
    sources[chunk["metadata"]["source"]] = sources.get(chunk["metadata"]["source"], 0) + 1

print(f"chunks          : {len(chunks)}")
print(f"chars per chunk : min {min(lengths)}, mean {sum(lengths) / len(lengths):.0f}, max {max(lengths)}")
print(f"per source      : {sources}")
print(f"suspiciously short chunks (<100 chars): {sum(1 for n in lengths if n < 100)}")
print(f"duplicate texts : {len(chunks) - len({c['text'] for c in chunks})}")

chunks          : 44
chars per chunk : min 71, mean 470, max 500
per source      : {'ai_course.pdf': 23, 'documentation.txt': 21}
suspiciously short chunks (<100 chars): 1
duplicate texts : 0


sources={
    "ai_course.pdf":3, 
    "documentation.txt":2
}

## 4. Chunking experiments


In [24]:
def ends_cleanly(text):
    """Does the chunk end at a sentence boundary rather than mid-word?"""
    return text.rstrip().endswith((".", "!", "?", ":", ";"))


def chunk_stats(documents, chunk_size, chunk_overlap, chunker=chunk_text):
    pieces = chunk_documents(documents, chunk_size, chunk_overlap, chunker)
    lengths = [len(c["text"]) for c in pieces]
    total_chars = sum(lengths)
    original = sum(len(d["text"]) for d in documents)
    return {
        "count": len(pieces),
        "avg": sum(lengths) / len(lengths),
        "min": min(lengths),
        "max": max(lengths),
        "clean_end_pct": 100 * sum(ends_cleanly(c["text"]) for c in pieces) / len(pieces),
        "duplication": total_chars / original,        # 1.0 = no repeated text
    }


print(f"{'chunk_size':>11} {'count':>7} {'avg':>7} {'min':>6} {'max':>6} {'clean end':>10}")
print("-" * 52)
for size in (200, 500, 1000):
    s = chunk_stats(documents, size, chunk_overlap=50)
    print(f"{size:>11} {s['count']:>7} {s['avg']:>7.0f} {s['min']:>6} {s['max']:>6} "
          f"{s['clean_end_pct']:>9.0f}%")

 chunk_size   count     avg    min    max  clean end
----------------------------------------------------
        200     126     196     60    200         7%
        500      44     470     71    500        14%
       1000      22     890    234   1000        18%


### Experiment 4.1 — read the chunks, not just the numbers


In [25]:
attention_doc = [d for d in documents if d["metadata"].get("page") == 3][0]

for size in (200, 500, 1000):
    pieces = chunk_text(attention_doc["text"], chunk_size=size, chunk_overlap=50)
    print("=" * 78)
    print(f"chunk_size = {size}   ->  {len(pieces)} chunks from this page")
    print("=" * 78)
    shown = pieces[min(1, len(pieces) - 1)]        # the 2nd chunk, if there is one
    print(shown[:size] + ("..." if len(shown) >= size else ""))
    print()

chunk_size = 200   ->  17 chunks from this page
ected into three vectors: a query, a key and a value. The relevance of token j to token i is the dot product between the query of i and the key of j. Those scores are divided by the square root of the...

chunk_size = 500   ->  6 chunks from this page
en used as weights to average the value vectors. This is called scaled dot product attention. Self attention means queries, keys and values all come from the same sequence, so the sequence attends to itself. Cross attention means the queries come from one sequence and the keys and values from another, which is how a decoder consults an encoder. Multi head attention runs several attention operations in parallel with different projections, so one head can track syntax while another tracks long ran...

chunk_size = 1000   ->  3 chunks from this page
ge topic information, and the results are concatenated and projected back. In a decoder, attention is causally masked: a token may only attend to i

### Experiment 4.2 — what overlap costs and what it buys


In [26]:
print(f"{'overlap':>8} {'count':>7} {'avg':>7} {'duplication':>12} {'clean end':>10}")
print("-" * 48)
for overlap in (0, 25, 50, 100, 200, 400):
    s = chunk_stats(documents, 500, overlap)
    print(f"{overlap:>8} {s['count']:>7} {s['avg']:>7.0f} {s['duplication']:>11.2f}x "
          f"{s['clean_end_pct']:>9.0f}%")

# And the setting that cannot work at all:
print("\nchunk_size=500 with chunk_overlap=500 (step would be 0):")
try:
    chunk_stats(documents, 500, 500)
except ValueError as error:
    print("  ValueError:", error)

 overlap   count     avg  duplication  clean end
------------------------------------------------
       0      39     480        1.00x        15%
      25      42     468        1.05x        17%
      50      44     470        1.10x        14%
     100      48     479        1.23x        12%
     200      62     486        1.61x        11%
     400     170     498        4.52x         6%

chunk_size=500 with chunk_overlap=500 (step would be 0):
  ValueError: chunk_overlap must be smaller than chunk_size


### Experiment 4.3 — a smarter chunker


In [27]:
def chunk_text_by_paragraph(text, chunk_size=500, chunk_overlap=50):
    """Pack whole paragraphs up to chunk_size; split only oversized paragraphs."""
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks, current = [], ""
    for paragraph in paragraphs:
        if len(paragraph) > chunk_size:                  # too big to pack: cut it
            if current:
                chunks.append(current)
                current = ""
            chunks.extend(chunk_text(paragraph, chunk_size, chunk_overlap))
            continue
        candidate = f"{current}\n\n{paragraph}" if current else paragraph
        if len(candidate) <= chunk_size:
            current = candidate                          # still fits, keep packing
        else:
            chunks.append(current)                       # flush and start a new chunk
            tail = current[-chunk_overlap:] if chunk_overlap else ""
            current = f"{tail} {paragraph}".strip() if tail else paragraph
    if current:
        chunks.append(current)
    return chunks


fixed = chunk_stats(documents, 500, 50, chunk_text)
para = chunk_stats(documents, 500, 50, chunk_text_by_paragraph)

print(f"{'chunker':>22} {'count':>7} {'avg':>7} {'min':>6} {'max':>6} {'clean end':>10}")
print("-" * 62)
for name, s in [("fixed size", fixed), ("paragraph aware", para)]:
    print(f"{name:>22} {s['count']:>7} {s['avg']:>7.0f} {s['min']:>6} {s['max']:>6} "
          f"{s['clean_end_pct']:>9.0f}%")
print("\nParagraph-aware chunks end cleanly far more often, at the cost of very")
print("uneven sizes. We keep fixed-size as the default so that the effect of")
print("chunk_size stays visible in Session 2 - but this is the one to reach for")
print("in production, and Session 3's challenges revisit it.")

               chunker   count     avg    min    max  clean end
--------------------------------------------------------------
            fixed size      44     470     71    500        14%
       paragraph aware      50     419     71    500        36%

Paragraph-aware chunks end cleanly far more often, at the cost of very
uneven sizes. We keep fixed-size as the default so that the effect of
chunk_size stays visible in Session 2 - but this is the one to reach for
in production, and Session 3's challenges revisit it.
